**Confirm the Frozen Search Instrument on the Test Split**

This notebook runs Experiment 4 of the experiment design (`./docs/experiment_design.pdf`, §4) as the manifest (`./docs/MANIFEST.md`, §6) specifies it: the search instrument frozen by `5_rank_and_freeze.ipynb` is run once on the sealed test split, beside the runner-up and the reference (the current design, L3 with K1). It refuses to start unless `./results/frozen_config.json` is committed to git and unchanged, so that the git history shows the freeze before any test file was read. The three pipelines are rebuilt end to end as a production service would run them: the archive index holds every person of both splits, the archive-derived dictionary D2 and the fine-tuned encoder E4 are rebuilt from the whole development split, the Daitch–Mokotoff look-ups run in PostgreSQL, and every query is encoded and answered from scratch while it is timed. It reads `./eval/`, `./dictionaries/`, `./results/frozen_config.json` and `./results/preflight.json`, and it outputs:

1. `./results/exp4_test_metrics.csv`: every metric of the three configurations on the test split with 95 % bootstrap intervals, overall, per query set and per stratum (section, script direction, name length and confidence band); the overall rows are also appended to `./results/metrics.csv` with `split = test`
2. `./results/exp4_mcnemar.json` and `./results/exp4_latency.json`: the exact McNemar tests of the selected configuration against the runner-up and the reference, and the cold-start and warm latencies
3. `./results/exp4_failures.csv`: every QS3 and QS4 query the selected configuration fails, with a heuristic cause flagged for human review
4. `./results/per_query/exp4_<config_id>.csv`: every test query's answer under each configuration

**1. Guard, Configuration and Data**

This cell first checks with git that `./results/frozen_config.json` is tracked and identical to the committed version, and stops otherwise. It then reads the frozen configurations and checks that this notebook implements every component they use (P3 placements of E1 or E4 with dictionary D3, the lexical generator L3, and the rankers K1 and K2), and reads the persons of both splits, the test queries and the curated dictionary.

1. `frozen`: the selected configuration, the runner-up and the reference, with their parameters chosen on all five development folds
2. `persons`, `queries`: every person (the index) and every test query, with the index tokens each query masks
3. the token arrays: every name token with its person, field and normalised form, each person's tokens of one field contiguous

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Guard, configuration and data  —  the test split, opened only after the freeze
#
# `git ls-files --error-unmatch` fails for an untracked file and `git diff
# --quiet HEAD` for a modified one, so either refusal means the freeze is
# not on record. The index holds every person: test queries meet the whole
# archive's distractors, as a visitor's query would.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import subprocess
import sys
import tempfile
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import psycopg
import torch
from anyascii import anyascii
from safetensors.numpy import load_file
from scipy.stats import binomtest

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
EVAL_DIR      = Path("eval")                       # persons and queries
RESULTS_DIR   = Path("results")                    # frozen config in, test results out
FROZEN_FILE   = RESULTS_DIR / "frozen_config.json"
PG_BIN        = Path(sys.executable).parent        # pg_ctl (conda env)
PG_DATA_DIR   = Path("cache/pgdata")               # the private cluster of 0_preflight
PG_PORT       = 5433                               # socket file name only; no TCP
SEED          = 333          # E4 fine-tuning and the bootstrap
QUERY_SETS    = ("QS1", "QS2", "QS3", "QS4", "QS5")
TOP_K         = 50           # persons kept per query
TOKEN_CAP     = 50           # P3 neighbours kept per query token
BOOTSTRAP     = 1000         # paired bootstrap resamples, clustered by target
LINK_COSINE   = 0.85         # failure analysis: encoder cosine that counts as linked
E4_BATCH, E4_LEARNING_RATE, E4_TEMPERATURE = 32, 2e-5, 0.07   # as in 3_compute_name_signals
E4_MAX_EPOCHS, E4_HELD_OUT_SHARE = 5, 0.10
RULE          = "─" * 72     # frames every printed report


def frozen_problems() -> list[str]:
    """Return why the frozen configuration is not on record in git (empty if it is)."""
    tracked = subprocess.run(["git", "ls-files", "--error-unmatch", str(FROZEN_FILE)],
                             capture_output=True, check=False).returncode == 0
    unchanged = subprocess.run(["git", "diff", "--quiet", "HEAD", "--", str(FROZEN_FILE)],
                               capture_output=True, check=False).returncode == 0
    return ([] if tracked else ["frozen_config.json is not committed"]) + (
        [] if unchanged else ["frozen_config.json differs from the committed version"])


if frozen_problems():
    raise RuntimeError("Experiment 4 is sealed: " + "; ".join(frozen_problems()))
FROZEN_COMMIT = subprocess.run(["git", "log", "-1", "--format=%H %cI", "--", str(FROZEN_FILE)],
                               capture_output=True, text=True, check=True).stdout.strip()


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    """Write `rows` to the CSV file `path`, with `columns` (default: the first row's keys)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns or list(rows[0]), lineterminator="\n",
                                extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


frozen = json.loads(FROZEN_FILE.read_text(encoding="utf-8"))
ROLES = {"selected": frozen["selected"], "runner_up": frozen["runner_up"],
         "reference": frozen["reference"]}
for _role, _entry in ROLES.items():
    for _spec in _entry["spec"].get("components") or [_entry["spec"]]:
        _supported = (_spec["generator"] == "L3" and not _spec["placement"]) or (
            _spec["placement"] == "P3" and _spec["generator"] in ("E1", "E4")
            and _spec["dictionary"] == "D3" and _spec["vocab"] == "archive+dict"
            and _spec["retrieval"] == "cutoff")
        assert _supported, f"{_role}: component {_spec['config_id']} is not implemented here"
    assert _entry["spec"]["ranker"] in ("K1", "K2"), f"{_role}: ranker not implemented here"
preflight = json.loads((RESULTS_DIR / "preflight.json").read_text(encoding="utf-8"))
persons = sorted(read_csv(EVAL_DIR / "persons_dev.csv") + read_csv(EVAL_DIR / "persons_test.csv"),
                 key=lambda row: row["person_id"])
person_ids = [row["person_id"] for row in persons]
person_index = {pid: i for i, pid in enumerate(person_ids)}
d1_forms = read_csv(EVAL_DIR / "d1_forms.csv")

# ── Vocabulary: archive forms (one display spelling each), then D1-only forms ──
_spellings = defaultdict(Counter)
for _row in persons:
    for _token in json.loads(_row["tokens"]):
        _spellings[(_token["script"], _token["norm"])][_token["display"]] += 1
forms = [(script, norm, sorted(c.items(), key=lambda item: (-item[1], item[0]))[0][0])
         for (script, norm), c in sorted(_spellings.items())]
N_ARCHIVE = len(forms)
form_index = {(script, norm): i for i, (script, norm, _) in enumerate(forms)}
for _form in d1_forms:
    if (_form["script"], _form["norm"]) not in form_index:
        form_index[(_form["script"], _form["norm"])] = len(forms)
        forms.append((_form["script"], _form["norm"], _form["display"]))

_tokens = sorted(((t["field"] == "surname", person_index[row["person_id"]],
                   form_index[(t["script"], t["norm"])], t["column"])
                  for row in persons for t in json.loads(row["tokens"])))
tok_surname, tok_person = np.array([t[0] for t in _tokens]), np.array([t[1] for t in _tokens])
tok_form, tok_column = np.array([t[2] for t in _tokens]), np.array([t[3] for t in _tokens])
seg_start = np.flatnonzero(np.r_[True, (np.diff(tok_person) != 0) | (np.diff(tok_surname) != 0)])
seg_person, seg_surname = tok_person[seg_start], tok_surname[seg_start]
form_token_count = np.bincount(tok_form, minlength=N_ARCHIVE)


@dataclass
class Query:
    """One test query: its row of queries_test.csv, tokens, target, relevant
    persons and the index tokens hidden from it."""

    row: dict
    tokens: list[dict]
    target: int
    relevant: set[int]
    masked: np.ndarray


queries = []
for _row in read_csv(EVAL_DIR / "queries_test.csv"):
    (_target,) = [person_index[p] for p in json.loads(_row["target_person_ids"])]
    _columns = sorted(filter(None, _row["masked_fields"].split(";")))
    queries.append(Query(_row, json.loads(_row["query_tokens"]), _target,
                         {person_index[p] for p in json.loads(_row["relevant_person_ids"])},
                         np.flatnonzero((tok_person == _target) & np.isin(tok_column, _columns))))
assert all(persons[q.target]["split"] == "test" for q in queries), "a test query targets dev"
query_set_index = np.array([QUERY_SETS.index(q.row["set"]) for q in queries])
print(f"✓ Guard passed (frozen_config.json committed: {FROZEN_COMMIT}); "
      f"{len(persons)} persons indexed, {len(queries)} test queries, {N_ARCHIVE} archive forms.")
for _role, _entry in ROLES.items():
    print(f"  {_role:<10} {_entry['spec']['config_id']:<22} {_entry['params']}")

✓ Guard passed (frozen_config.json committed: 9ea5794ba47fd32718a88360188a040ebb852c73 2026-10-10T21:17:57-04:00); 637 persons indexed, 415 test queries, 1206 archive forms.
  selected   exp3_K2_sum            {'lam': 0.8, 'tau_d': 0.7500001788139343}
  runner_up  exp3_K2_product        {'lam': 0.9, 'tau_d': 0.6250002235174357}
  reference  exp1_L3                {'c': None, 'tau_link': None, 'lam': 1.0, 'tau_d': 0.525}


**2. Encoders and the Index's Embeddings**

This cell prepares the two embedding models the frozen components use, as an ingestion step would. E1 runs through the NumPy forward pass of the manifest's Appendix A. E4 is E1 fine-tuned once more, now on the pairs of the whole development split (a Latin token and a Hebrew token of the same field on the same person), with the procedure of `3_compute_name_signals.ipynb`: InfoNCE over in-batch negatives, at most five epochs, early stopping on a held-out tenth. Its weights are exported to NumPy, so that both models run through the same forward pass. It writes the ingestion artefacts that a cold start loads:

1. `./cache/exp4_e4_weights.npz`: the fine-tuned E4 weights
2. `./cache/exp4_index_vectors.npz`: every vocabulary form embedded by E1 and by E4

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Encoders and the index's embeddings  —  E1, and E4 fine-tuned on all of dev
#
# The test persons' tokens never enter E4's training pairs; they only
# receive embeddings, as any newly ingested record would.
# ─────────────────────────────────────────────────────────────────────────────
if "queries" not in globals():
    raise RuntimeError("Run the 'Guard, Configuration and Data' cell first.")


def layer_norm(x: np.ndarray, gain: np.ndarray, bias: np.ndarray) -> np.ndarray:
    """Normalise the last axis of `x`, then scale by `gain` and shift by `bias`."""
    mean = x.mean(-1, keepdims=True)
    return (x - mean) / np.sqrt(((x - mean) ** 2).mean(-1, keepdims=True) + 1e-5) * gain + bias


def e1_attention(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the 8-head self-attention update of layer `prefix` for the states `x`."""
    heads, head_dim = 8, x.shape[1] // 8
    h = layer_norm(x, w[prefix + "norm1.weight"], w[prefix + "norm1.bias"])
    qkv = h @ w[prefix + "self_attn.in_proj_weight"].T + w[prefix + "self_attn.in_proj_bias"]
    q, k, v = (t.reshape(len(x), heads, head_dim).transpose(1, 0, 2) for t in np.split(qkv, 3, -1))
    scores = np.exp(q @ k.transpose(0, 2, 1) / np.sqrt(head_dim))
    attended = ((scores / scores.sum(-1, keepdims=True)) @ v).transpose(1, 0, 2).reshape(x.shape)
    out_proj = prefix + "self_attn.out_proj."
    return attended @ w[out_proj + "weight"].T + w[out_proj + "bias"]


def e1_feed_forward(w: dict, x: np.ndarray, prefix: str) -> np.ndarray:
    """Return the ReLU feed-forward update of layer `prefix` for the states `x`."""
    h = layer_norm(x, w[prefix + "norm2.weight"], w[prefix + "norm2.bias"])
    hidden = np.maximum(0, h @ w[prefix + "linear1.weight"].T + w[prefix + "linear1.bias"])
    return hidden @ w[prefix + "linear2.weight"].T + w[prefix + "linear2.bias"]


def e1_encode(w: dict, text: str) -> np.ndarray:
    """Return the unit-length embedding of `text` under E1's architecture with weights `w`."""
    ids = list(text.encode("utf-8"))[:256]
    x = w["embedding.weight"][ids] + w["pos_embedding.weight"][:len(ids)]
    for layer in range(6):
        x = x + e1_attention(w, x, f"transformer.layers.{layer}.")
        x = x + e1_feed_forward(w, x, f"transformer.layers.{layer}.")
    pooled = x.mean(0)
    return pooled / np.linalg.norm(pooled)


class TorchByteEncoder(torch.nn.Module):
    """E1's byte-level encoder in PyTorch, for fine-tuning E4."""

    def __init__(self, weights: dict):
        """Build the encoder and load the NumPy `weights` of E1's checkpoint."""
        super().__init__()
        self.embedding = torch.nn.Embedding(256, 256)
        self.pos_embedding = torch.nn.Embedding(weights["pos_embedding.weight"].shape[0], 256)
        layer = torch.nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=1024,
                                                 dropout=0.1, batch_first=True, norm_first=True)
        self.transformer = torch.nn.TransformerEncoder(layer, num_layers=6,
                                                       enable_nested_tensor=False)
        self.load_state_dict({name: torch.from_numpy(value.copy())
                              for name, value in weights.items()})

    def forward(self, texts: list[str]) -> torch.Tensor:
        """Return the unit-length embeddings of `texts`, one row per text."""
        ids = [list(text.encode("utf-8"))[:256] for text in texts]
        length = max(len(row) for row in ids)
        batch = torch.tensor([row + [0] * (length - len(row)) for row in ids])
        real = torch.tensor([[i < len(row) for i in range(length)] for row in ids])
        x = self.embedding(batch) + self.pos_embedding(torch.arange(length)).unsqueeze(0)
        x = self.transformer(x, src_key_padding_mask=~real)
        pooled = (x * real.unsqueeze(-1)).sum(1) / real.sum(1, keepdim=True)
        return torch.nn.functional.normalize(pooled, dim=-1)


def info_nce(model: TorchByteEncoder, batch: list[tuple[str, str]]) -> torch.Tensor:
    """Return the InfoNCE loss of a batch of (Latin, Hebrew) pairs."""
    anchors, positives = model([a for a, _ in batch]), model([b for _, b in batch])
    logits = anchors @ positives.T / E4_TEMPERATURE
    return torch.nn.functional.cross_entropy(logits, torch.arange(len(batch)))


def held_out_loss(model: TorchByteEncoder, pairs: list[tuple[str, str]]) -> float:
    """Return the mean InfoNCE loss over `pairs`, without gradients."""
    model.eval()
    with torch.no_grad():
        return float(np.mean([info_nce(model, pairs[i:i + E4_BATCH]).item()
                              for i in range(0, len(pairs), E4_BATCH)
                              if len(pairs[i:i + E4_BATCH]) > 1]))


def development_pairs() -> list[tuple[str, str]]:
    """Return every (Latin, Hebrew) display pair of one field on one development person."""
    pairs = set()
    for row in persons:
        if row["split"] != "dev":
            continue
        tokens = json.loads(row["tokens"])
        for field_name in ("given", "surname"):
            in_field = [t for t in tokens if t["field"] == field_name]
            pairs.update((a["display"], b["display"]) for a in in_field for b in in_field
                         if a["script"] == "latin" and b["script"] == "hebrew")
    return sorted(pairs)


def fine_tune_e4(weights: dict) -> tuple[dict, dict]:
    """Fine-tune E1 on every development pair; return NumPy weights and a report."""
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    pairs = development_pairs()
    order = rng.permutation(len(pairs))
    held_out = [pairs[i] for i in order[:round(len(pairs) * E4_HELD_OUT_SHARE)]]
    train = [pairs[i] for i in order[len(held_out):]]
    model = TorchByteEncoder(weights)
    optimiser = torch.optim.AdamW(model.parameters(), lr=E4_LEARNING_RATE)
    best = {"epoch": 0, "loss": held_out_loss(model, held_out),
            "state": {k: v.clone() for k, v in model.state_dict().items()}}
    report = {"pairs": len(pairs), "loss_before": best["loss"]}
    for epoch in range(1, E4_MAX_EPOCHS + 1):
        model.train()
        for start in range(0, len(train), E4_BATCH):
            batch = [train[i] for i in rng.permutation(len(train))[start:start + E4_BATCH]]
            if len(batch) > 1:
                optimiser.zero_grad()
                info_nce(model, batch).backward()
                optimiser.step()
        loss = held_out_loss(model, held_out)
        if loss < best["loss"]:
            best = {"epoch": epoch, "loss": loss,
                    "state": {k: v.clone() for k, v in model.state_dict().items()}}
    return ({k: v.numpy() for k, v in best["state"].items()},
            report | {"epoch_kept": best["epoch"], "loss_after": best["loss"]})


ENCODER_WEIGHTS = {"E1": load_file(preflight["E1"]["checkpoint"])}
ENCODER_WEIGHTS["E4"], e4_report = fine_tune_e4(ENCODER_WEIGHTS["E1"])
np.savez(Path("cache/exp4_e4_weights.npz"), **ENCODER_WEIGHTS["E4"])
index_vectors = {name: np.stack([e1_encode(w, display) for _, _, display in forms])
                 for name, w in ENCODER_WEIGHTS.items()}
np.savez(Path("cache/exp4_index_vectors.npz"), **index_vectors)
print(f"{RULE}\nEncoders: {len(forms)} vocabulary forms embedded\n{RULE}")
print(f"  OK E1   checkpoint {preflight['E1']['checkpoint']}")
print(f"  OK E4   {e4_report['pairs']} development pairs, epoch kept {e4_report['epoch_kept']}, "
      f"held-out loss {e4_report['loss_before']:.3f} → {e4_report['loss_after']:.3f}")
_pair = {name: float(e1_encode(w, "Rebeca") @ e1_encode(w, "רבקה"))
         for name, w in ENCODER_WEIGHTS.items()}
print(f"{RULE}\n  cos(Rebeca, רבקה): E1 {_pair['E1']:.3f}, E4 {_pair['E4']:.3f}")

────────────────────────────────────────────────────────────────────────
Encoders: 1627 vocabulary forms embedded
────────────────────────────────────────────────────────────────────────
  OK E1   checkpoint assets/e1_repo/checkpoints/best_v2/model.safetensors
  OK E4   607 development pairs, epoch kept 3, held-out loss 1.681 → 1.445
────────────────────────────────────────────────────────────────────────
  cos(Rebeca, רבקה): E1 0.889, E4 0.784


**3. Dictionaries, Links and the Daitch–Mokotoff Table**

This cell builds the rest of the index as an ingestion step would:

1. **D2**, rebuilt from the whole development split: the Spanish–Hebrew pairs of given-name tokens and of surname tokens recorded on the same person, kept at a support of at least two persons and merged into groups by connected components; **D3**, D2 merged with the curated D1
2. **Links**: every archive form joins the D3 groups of its own form, or else those of its nearest D3 form by the component's encoder if the cosine reaches the component's frozen `τ_link`
3. **PostgreSQL**: a table of the archive forms with their Daitch–Mokotoff codes (Hebrew romanised with AnyAscii) under a GIN index, which L3 queries for every query token

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Dictionaries, links and the D–M table  —  ingestion of the whole archive
# ─────────────────────────────────────────────────────────────────────────────
if "index_vectors" not in globals():
    raise RuntimeError("Run the 'Encoders' cell first.")


def connected_groups(groups: list[set[int]]) -> dict[str, set[int]]:
    """Merge groups that share a form (connected components)."""
    merged, owner = [], {}
    for members in groups:
        overlapping = {owner[f] for f in members if f in owner}
        union = set(members).union(*(merged[i] for i in overlapping))
        for i in overlapping:
            merged[i] = set()
        merged.append(union)
        for form in union:
            owner[form] = len(merged) - 1
    return {f"D3_{i}": m for i, m in enumerate(merged) if m}


def d2_pairs() -> list[set[int]]:
    """Return D2's pairs: Latin–Hebrew token pairs on ≥ 2 development persons."""
    support = Counter()
    for row in persons:
        if row["split"] == "dev":
            tokens = json.loads(row["tokens"])
            for field_name in ("given", "surname"):
                pick = {s: {form_index[(t["script"], t["norm"])] for t in tokens
                            if t["field"] == field_name and t["script"] == s}
                        for s in ("latin", "hebrew")}
                support.update((a, b) for a in pick["latin"] for b in pick["hebrew"])
    return [{a, b} for (a, b), count in support.items() if count >= 2]


def build_links(generator: str, tau_link: float) -> tuple[dict, list[set[str]]]:
    """Link every archive form to D3 groups under one encoder and threshold.

    Returns:
        Each group's linked archive forms, and each archive form's groups.
    """
    dict_forms = np.array(sorted(D3_GROUPS_OF))
    cosines = index_vectors[generator][:N_ARCHIVE] @ index_vectors[generator][dict_forms].T
    members, groups_of_form = defaultdict(list), []
    for form in range(N_ARCHIVE):
        groups = D3_GROUPS_OF.get(form, set())
        best = int(np.argmax(cosines[form]))
        if not groups and cosines[form, best] >= tau_link:
            groups = D3_GROUPS_OF[dict_forms[best]]
        groups_of_form.append(groups)
        for group in groups:
            members[group].append(form)
    return {g: np.array(f) for g, f in members.items()}, groups_of_form


def start_postgres(data_dir: Path) -> Path:
    """Start the private PostgreSQL cluster; return its socket folder."""
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-m", "fast", "stop"],
                   check=False, capture_output=True)
    socket_dir = Path(tempfile.mkdtemp(prefix="pg"))
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-l", data_dir / "server.log", "-w",
                    "-o", f"-k {socket_dir} -c listen_addresses= -p {PG_PORT}", "start"],
                   check=True, capture_output=True)
    return socket_dir


_d1_groups = defaultdict(set)
for _form in d1_forms:
    _d1_groups[_form["group_id"]].add(form_index[(_form["script"], _form["norm"])])
D3_GROUPS = connected_groups([*_d1_groups.values(), *d2_pairs()])
D3_GROUPS_OF = defaultdict(set)
for _group, _members in D3_GROUPS.items():
    for _form in _members:
        D3_GROUPS_OF[_form].add(_group)
LINKS = {}
for _entry in ROLES.values():
    for _spec in _entry["spec"].get("components", []):
        if _spec["placement"] == "P3":
            _theta = _entry["component_params"][_spec["config_id"]]
            LINKS[(_spec["generator"], _theta["tau_link"])] = build_links(_spec["generator"],
                                                                         _theta["tau_link"])

connection = psycopg.connect(host=str(start_postgres(PG_DATA_DIR)), port=PG_PORT,
                             user="postgres", dbname="postgres", autocommit=True)
connection.execute("DROP TABLE IF EXISTS archive_forms")
connection.execute("CREATE TABLE archive_forms (form_id int, script text, norm text, dm text[])")
with connection.cursor().copy("COPY archive_forms (form_id, script, norm) FROM STDIN") as _copy:
    for _i, (_script, _norm, _) in enumerate(forms[:N_ARCHIVE]):
        _copy.write_row((_i, _script, anyascii(_norm) if _script == "hebrew" else _norm))
connection.execute("UPDATE archive_forms SET dm = array_remove(daitch_mokotoff(norm), '000000')")
connection.execute("CREATE INDEX archive_forms_dm ON archive_forms USING gin (dm)")
print(f"{RULE}\nIndex: D3 and links\n{RULE}")
print(f"  D3   {len(D3_GROUPS)} groups ({len(_d1_groups)} curated, {len(d2_pairs())} D2 pairs)")
for (_generator, _tau), (_members, _groups_of_form) in LINKS.items():
    print(f"  {_generator}   τ_link {_tau}: {sum(bool(g) for g in _groups_of_form)} of "
          f"{N_ARCHIVE} archive forms linked")
print(f"{RULE}\n  PostgreSQL: archive_forms with D–M codes and a GIN index")

────────────────────────────────────────────────────────────────────────
Index: D3 and links
────────────────────────────────────────────────────────────────────────
  D3   131 groups (105 curated, 74 D2 pairs)
  E4   τ_link 0.9: 478 of 1206 archive forms linked
  E1   τ_link 0.85: 561 of 1206 archive forms linked
────────────────────────────────────────────────────────────────────────
  PostgreSQL: archive_forms with D–M codes and a GIN index


**4. The Frozen Pipelines and Their Latency**

This cell runs each frozen configuration as a service would answer a visitor: every query token is encoded afresh (no cached query embedding), each component finds its matches (P3: the token's neighbours among the archive's and the dictionary's forms, whose groups bring their linked forms; L3: a PostgreSQL look-up of the token's Daitch–Mokotoff codes), the ranker combines them (K2: each index token's best similarity, dictionary-derived matches weighted by λ; K1: plain similarity), the given-name and surname scores are aggregated per person and the persons are ranked. Each query is timed from start to finish, warm, after `WARM_UP` untimed queries. The cold start is timed separately: loading E1's checkpoint, E4's weights and the index's vectors from disk, then answering the first query. It produces `answers`, each configuration's 50 best persons per query with their scores and times, and `latency`.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# The frozen pipelines  —  end to end, timed per query
# ─────────────────────────────────────────────────────────────────────────────
if "connection" not in globals():
    raise RuntimeError("Run the 'Dictionaries, Links and the Daitch–Mokotoff Table' cell first.")

WARM_UP = 20   # untimed queries before the warm timing


def p3_neighbours(spec: dict, theta: dict, token_vector: np.ndarray, query: Query):
    """Return the token's P3 neighbours (cosine ≥ c, at most 50) among the archive
    forms with an unmasked token and the D3 forms: vocabulary indices and cosines."""
    available = form_token_count > np.bincount(tok_form[query.masked], minlength=N_ARCHIVE)
    vocabulary = np.union1d(np.flatnonzero(available), sorted(D3_GROUPS_OF))
    cosines = np.clip(index_vectors[spec["generator"]][vocabulary] @ token_vector, 0, None)
    order = np.argsort(-cosines, kind="stable")
    keep = order[cosines[order] >= theta["c"]][:TOKEN_CAP]
    return vocabulary[keep], cosines[keep]


def derived_from_seeds(seeds: list[tuple[float, set[str]]], members: dict) -> np.ndarray:
    """Give every archive form linked to a seed's groups the seed's similarity (the best)."""
    derived = np.zeros(N_ARCHIVE)
    for similarity, groups in seeds:
        for group in groups:
            group_forms = members.get(group, np.zeros(0, int))
            derived[group_forms] = np.maximum(derived[group_forms], similarity)
    return derived


def p3_similarity(spec: dict, theta: dict, token_vector: np.ndarray, token: dict,
                  query: Query) -> tuple[np.ndarray, np.ndarray]:
    """Return a P3 component's direct and derived similarity to every archive form:
    neighbours are direct matches (archive forms) and seed their groups, as does
    the token's own form when it belongs to D3."""
    members, groups_of_form = LINKS[(spec["generator"], theta["tau_link"])]
    own = form_index.get((token["script"], token["norm"]))
    seeds = [(1.0, D3_GROUPS_OF[own])] if own in D3_GROUPS_OF else []
    direct = np.zeros(N_ARCHIVE)
    for form, cosine in zip(*p3_neighbours(spec, theta, token_vector, query)):
        if form < N_ARCHIVE:
            direct[form] = cosine
        seeds.append((float(cosine), groups_of_form[form] if form < N_ARCHIVE
                      else D3_GROUPS_OF.get(form, set())))
    return direct, derived_from_seeds(seeds, members)


def l3_similarity(token: dict) -> np.ndarray:
    """Return 1 for every archive form sharing a D–M code with `token` (PostgreSQL)."""
    text = anyascii(token["norm"]) if token["script"] == "hebrew" else token["norm"]
    similarity = np.zeros(N_ARCHIVE)
    found = connection.execute("SELECT form_id FROM archive_forms WHERE dm && "
                               "array_remove(daitch_mokotoff(%s), '000000')", (text,)).fetchall()
    similarity[[row[0] for row in found]] = 1.0
    return similarity


def token_similarities(entry: dict, query: Query, encoded: dict) -> np.ndarray:
    """Return, per query token, every index token's similarity under the entry's ranker."""
    components = entry["spec"].get("components") or [entry["spec"]]
    lam = entry["params"].get("lam", 1.0)
    rows = []
    for i, token in enumerate(query.tokens):
        best = np.zeros(N_ARCHIVE)
        for spec in components:
            if spec["placement"] == "P3":
                direct, derived = p3_similarity(spec, entry["component_params"][spec["config_id"]],
                                                encoded[spec["generator"]][i], token, query)
                best = np.maximum(best, np.maximum(direct, lam * derived))
            else:
                best = np.maximum(best, l3_similarity(token))
        row = best[tok_form]
        row[query.masked] = 0
        rows.append(row)
    return np.array(rows)


def person_scores(entry: dict, similarities: np.ndarray) -> np.ndarray:
    """Aggregate token similarities into person scores (sum halved, product or mean)."""
    maxima = np.maximum.reduceat(similarities, seg_start, axis=1)
    given, surname = (np.zeros((len(similarities), len(persons))) for _ in range(2))
    given[:, seg_person[~seg_surname]] = maxima[:, ~seg_surname]
    surname[:, seg_person[seg_surname]] = maxima[:, seg_surname]
    if len(given) == 1:
        return np.maximum(given[0], surname[0])
    if entry["spec"]["aggregation"] == "product":
        pairs = given[:, None, :] * surname[None, :, :]
    else:
        pairs = (given[:, None, :] + surname[None, :, :]) / 2
    pairs[np.arange(len(given)), np.arange(len(given))] = 0
    return pairs.max(axis=(0, 1))


def answer(entry: dict, query: Query, weights: dict) -> tuple[np.ndarray, np.ndarray]:
    """Answer one query end to end: encode, match, rank. Returns the 50 best persons."""
    generators = {s["generator"] for s in entry["spec"].get("components", [])
                  if s["placement"] == "P3"}
    encoded = {g: [e1_encode(weights[g], t["display"]) for t in query.tokens] for g in generators}
    scores = person_scores(entry, token_similarities(entry, query, encoded))
    candidates = np.flatnonzero(scores > 0)
    ranked = candidates[np.lexsort((candidates, -scores[candidates]))][:TOP_K]
    ids, values = np.full(TOP_K, -1), np.zeros(TOP_K)
    ids[:len(ranked)], values[:len(ranked)] = ranked, scores[ranked]
    return ids, values


def cold_start(entry: dict) -> dict:
    """Time loading the models and index from disk and answering the first query."""
    started = time.perf_counter()
    weights = {"E1": load_file(preflight["E1"]["checkpoint"]),
               "E4": dict(np.load(Path("cache/exp4_e4_weights.npz")))}
    dict(np.load(Path("cache/exp4_index_vectors.npz")))
    loaded = time.perf_counter()
    answer(entry, queries[0], weights)
    return {"load_ms": (loaded - started) * 1000,
            "first_query_ms": (time.perf_counter() - loaded) * 1000}


answers, latency = {}, {}
for _role, _entry in ROLES.items():
    latency[_role] = {"cold": cold_start(_entry)}
    for _query in queries[:WARM_UP]:
        answer(_entry, _query, ENCODER_WEIGHTS)
    _ids, _scores, _ms = [], [], []
    for _query in queries:
        _started = time.perf_counter()
        _top = answer(_entry, _query, ENCODER_WEIGHTS)
        _ms.append((time.perf_counter() - _started) * 1000)
        _ids.append(_top[0])
        _scores.append(_top[1])
    answers[_role] = {"ids": np.array(_ids), "scores": np.array(_scores), "ms": np.array(_ms)}
    latency[_role]["warm_p50_ms"] = float(np.percentile(_ms, 50))
    latency[_role]["warm_p95_ms"] = float(np.percentile(_ms, 95))
connection.close()
subprocess.run([PG_BIN / "pg_ctl", "-D", PG_DATA_DIR, "-m", "fast", "stop"], check=False,
               capture_output=True)

print(f"{RULE}\nLatency on the test queries (CPU: this machine)\n{RULE}")
for _role, _times in latency.items():
    print(f"  {_role:<10} cold: load {_times['cold']['load_ms']:7.1f} ms + first query "
          f"{_times['cold']['first_query_ms']:6.1f} ms   warm: p50 "
          f"{_times['warm_p50_ms']:6.1f} ms, p95 {_times['warm_p95_ms']:6.1f} ms")
print(f"{RULE}\n  {len(queries)} test queries answered by each of {len(ROLES)} configurations")

────────────────────────────────────────────────────────────────────────
Latency on the test queries (CPU: this machine)
────────────────────────────────────────────────────────────────────────
  selected   cold: load    45.9 ms + first query   84.4 ms   warm: p50   67.7 ms, p95  110.5 ms
  runner_up  cold: load    56.6 ms + first query   72.7 ms   warm: p50   67.9 ms, p95  115.0 ms
  reference  cold: load    48.3 ms + first query    0.5 ms   warm: p50    0.2 ms, p95    0.3 ms
────────────────────────────────────────────────────────────────────────
  415 test queries answered by each of 3 configurations


**5. Metrics, Intervals and McNemar Tests**

This cell measures the three configurations on the test queries at their frozen display cut-offs `τ_d`:

1. per query: R@10, the reciprocal rank of the first relevant person, unrelated@10, whether the list is empty, and whether the target is among the first ten shown (the hit the McNemar test compares)
2. overall (the macro-average over QS1–QS5), per query set and per stratum: section, script of the query, name length (shortest query token of four letters or fewer, or longer) and the target's extraction confidence (85–94 or 95–100), each with a 95 % interval from `BOOTSTRAP` paired resamples clustered by target person
3. exact McNemar tests on the hits of the selected configuration against the runner-up and against the reference

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Metrics, intervals and McNemar tests  —  the test split, measured once
#
# A stratum's R@10 is the plain mean over its queries; the overall row is
# the macro-average over query sets, the primary score of the design.
# ─────────────────────────────────────────────────────────────────────────────
if "answers" not in globals():
    raise RuntimeError("Run the 'Frozen Pipelines' cell first.")

STRATA = {
    "query_set": [q.row["set"] for q in queries],
    "section": [q.row["section"] for q in queries],
    "query_script": [q.row["query_script"] for q in queries],
    "name_length": ["≤ 4" if min(len(t["norm"]) for t in q.tokens) <= 4 else "> 4"
                    for q in queries],
    "confidence": ["85–94" if int(persons[q.target]["confidence"]) < 95 else "95–100"
                   for q in queries],
}


def per_query(role: str) -> dict[str, np.ndarray]:
    """Return each test query's metrics under one configuration at its τ_d."""
    tau, result = ROLES[role]["params"]["tau_d"], defaultdict(list)
    for query, ids, scores in zip(queries, answers[role]["ids"], answers[role]["scores"]):
        shown = [p for p, s in zip(ids, scores) if p >= 0 and s > 0 and s >= tau]
        relevant = [rank + 1 for rank, p in enumerate(shown) if p in query.relevant]
        found = sum(p in query.relevant for p in shown[:10])
        result["r10"].append(found / min(10, len(query.relevant)))
        result["rr"].append(1 / relevant[0] if relevant else 0.0)
        result["unrelated"].append(min(10, len(shown)) - found)
        result["empty"].append(float(not shown))
        result["hit"].append(float(query.target in shown[:10]))
    return {name: np.array(values) for name, values in result.items()}


_clusters = np.unique([q.target for q in queries], return_inverse=True)[1]
_draws = np.random.default_rng(SEED).integers(0, _clusters.max() + 1,
                                              size=(BOOTSTRAP, _clusters.max() + 1))
WEIGHTS = np.stack([np.bincount(d, minlength=_clusters.max() + 1) for d in _draws])[:, _clusters]


def macro_with_interval(values: np.ndarray, rows: np.ndarray) -> tuple[float, float, float]:
    """Return the macro-average over query sets of `values` on `rows`, and its
    95 % bootstrap interval."""
    in_sets = [rows & (query_set_index == s) for s in range(len(QUERY_SETS))]
    in_sets = [mask for mask in in_sets if mask.any()]
    point = np.mean([values[mask].mean() for mask in in_sets])
    resampled = np.mean([WEIGHTS[:, mask] @ values[mask] / WEIGHTS[:, mask].sum(axis=1).clip(1e-9)
                         for mask in in_sets], axis=0)
    return float(point), *map(float, np.percentile(resampled, [2.5, 97.5]))


def mean_with_interval(values: np.ndarray, rows: np.ndarray) -> tuple[float, float, float]:
    """Return the plain mean of `values` on `rows`, and its 95 % bootstrap interval."""
    resampled = WEIGHTS[:, rows] @ values[rows] / WEIGHTS[:, rows].sum(axis=1).clip(1e-9)
    return float(values[rows].mean()), *map(float, np.percentile(resampled, [2.5, 97.5]))


def mcnemar(first: np.ndarray, second: np.ndarray) -> dict:
    """Return the exact McNemar test of two configurations' paired hits."""
    only_first, only_second = int(np.sum((first == 1) & (second == 0))), int(
        np.sum((first == 0) & (second == 1)))
    discordant = only_first + only_second
    p_value = binomtest(only_first, discordant, 0.5).pvalue if discordant else 1.0
    return {"selected_only": only_first, "other_only": only_second, "p_value": float(p_value)}


query_metrics = {role: per_query(role) for role in ROLES}
test_rows = []
for _role, _metrics in query_metrics.items():
    _all = np.ones(len(queries), bool)
    _strata = [("overall", "macro", _all)] + [
        (name, value, np.array(values) == value)
        for name, values in STRATA.items() for value in sorted(set(values))]
    for _stratum, _value, _rows in _strata:
        _summary = macro_with_interval if _stratum == "overall" else mean_with_interval
        _r10, _rr = _summary(_metrics["r10"], _rows), _summary(_metrics["rr"], _rows)
        test_rows.append({
            "role": _role, "config_id": ROLES[_role]["spec"]["config_id"], "stratum": _stratum,
            "value": _value, "n_queries": int(_rows.sum()), "r_at_10": _r10[0],
            "r_at_10_lo": _r10[1], "r_at_10_hi": _r10[2], "mrr": _rr[0], "mrr_lo": _rr[1],
            "mrr_hi": _rr[2], "unrelated_at_10": float(_metrics["unrelated"][_rows].mean()),
            "empty_rate": float(_metrics["empty"][_rows].mean()),
            "hit_rate": float(_metrics["hit"][_rows].mean()),
            "latency_p50_ms": float(np.percentile(answers[_role]["ms"][_rows], 50)),
            "latency_p95_ms": float(np.percentile(answers[_role]["ms"][_rows], 95))})
mcnemar_tests = {other: mcnemar(query_metrics["selected"]["hit"], query_metrics[other]["hit"])
                 for other in ("runner_up", "reference")}

print(f"{RULE}\nTest split: macro R@10 [95 % CI], MRR, unrelated@10, empty rate\n{RULE}")
for _row in (r for r in test_rows if r["stratum"] == "overall"):
    print(f"  {_row['role']:<10} {_row['config_id']:<18} R@10 {_row['r_at_10']:.3f} "
          f"[{_row['r_at_10_lo']:.3f}, {_row['r_at_10_hi']:.3f}]  MRR {_row['mrr']:.3f}  "
          f"unrelated {_row['unrelated_at_10']:.2f}  empty {_row['empty_rate']:.2f}")
for _other, _test in mcnemar_tests.items():
    print(f"  McNemar selected vs {_other:<10} {_test['selected_only']:>3} vs "
          f"{_test['other_only']:>3} discordant hits, exact p = {_test['p_value']:.4f}")
print(f"{RULE}\n  {len(test_rows)} rows over {len(STRATA)} strata")

────────────────────────────────────────────────────────────────────────
Test split: macro R@10 [95 % CI], MRR, unrelated@10, empty rate
────────────────────────────────────────────────────────────────────────
  selected   exp3_K2_sum        R@10 0.869 [0.821, 0.910]  MRR 0.749  unrelated 3.71  empty 0.03
  runner_up  exp3_K2_product    R@10 0.872 [0.824, 0.913]  MRR 0.750  unrelated 3.82  empty 0.03
  reference  exp1_L3            R@10 0.459 [0.416, 0.508]  MRR 0.429  unrelated 0.37  empty 0.43
  McNemar selected vs runner_up    0 vs   1 discordant hits, exact p = 1.0000
  McNemar selected vs reference  159 vs   1 discordant hits, exact p = 0.0000
────────────────────────────────────────────────────────────────────────
  42 rows over 5 strata


**6. Error Analysis**

This cell lists every QS3 and QS4 query whose target the selected configuration does not show among its first ten persons, the cross-script queries the dictionaries and embeddings exist for. Each failure gets a heuristic cause, which a person should confirm:

1. `ranking_error`: the target is among the configuration's candidates but ranked below the first ten shown, or under the display cut-off
2. `missing_equivalence`: the target is not a candidate, its surname is linked to the query's surname (shared Daitch–Mokotoff code, or E1 cosine of at least `LINK_COSINE`) but its given name is not linked to the query's given name in any of these ways or through a D3 group, as with civil names that only echo the Hebrew one
3. `transcription_error`: the target is not a candidate and even its surname is not linked, which for a transliterated surname suggests a misreading, or its extraction confidence is below 90
4. `other`: anything else

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Error analysis  —  failed QS3/QS4 queries of the selected configuration
#
# The links are re-checked with Abydos-free signals already loaded: the
# D–M codes from PostgreSQL's table (fetched again here) and E1's cosine.
# ─────────────────────────────────────────────────────────────────────────────
if "query_metrics" not in globals():
    raise RuntimeError("Run the 'Metrics' cell first.")

_connection = psycopg.connect(host=str(start_postgres(PG_DATA_DIR)), port=PG_PORT,
                              user="postgres", dbname="postgres", autocommit=True)
DM_OF_TEXT = {}


def dm_codes(token: dict) -> set[str]:
    """Return a token's Daitch–Mokotoff codes (PostgreSQL), Hebrew romanised first."""
    text = anyascii(token["norm"]) if token["script"] == "hebrew" else token["norm"]
    if text not in DM_OF_TEXT:
        DM_OF_TEXT[text] = set(_connection.execute("SELECT daitch_mokotoff(%s)",
                                                   (text,)).fetchone()[0] or []) - {"000000"}
    return DM_OF_TEXT[text]


def linked(first: dict, second: dict) -> bool:
    """Return True if two tokens share a D–M code, reach LINK_COSINE under E1, or
    share a D3 group."""
    groups = [D3_GROUPS_OF.get(form_index.get((t["script"], t["norm"])), set())
              for t in (first, second)]
    cosine = float(e1_encode(ENCODER_WEIGHTS["E1"], first["display"])
                   @ e1_encode(ENCODER_WEIGHTS["E1"], second["display"]))
    return bool(dm_codes(first) & dm_codes(second)) or cosine >= LINK_COSINE or bool(
        groups[0] & groups[1])


def failure_cause(query: Query, ids: np.ndarray) -> str:
    """Return the heuristic cause of one failed cross-script query."""
    if query.target in ids:
        return "ranking_error"
    target_tokens = [t for t in json.loads(persons[query.target]["tokens"])
                     if t["column"] not in query.row["masked_fields"].split(";")]
    field_linked = {name: any(linked(q, t) for q in query.tokens if q["field"] == name
                              for t in target_tokens if t["field"] == name)
                    for name in ("given", "surname")}
    if field_linked["surname"] and not field_linked["given"]:
        return "missing_equivalence"
    if not field_linked["surname"] or int(persons[query.target]["confidence"]) < 90:
        return "transcription_error"
    return "other"


def display_names(index: int) -> str:
    """Return a person's recorded names, Latin then Hebrew script."""
    tokens = json.loads(persons[index]["tokens"])
    scripts = [s for s in ("latin", "hebrew") if any(t["script"] == s for t in tokens)]
    return " / ".join(" ".join(t["display"] for t in tokens if t["script"] == script)
                      for script in scripts)


failures = []
for _query, _ids, _hit in zip(queries, answers["selected"]["ids"],
                              query_metrics["selected"]["hit"]):
    if _query.row["set"] in ("QS3", "QS4") and not _hit:
        failures.append({"query_id": _query.row["query_id"], "query_text": _query.row["query_text"],
                         "target_names": display_names(_query.target),
                         "top3_returned": " | ".join(display_names(p) for p in _ids[:3] if p >= 0),
                         "cause": failure_cause(_query, _ids), "review": "pending"})
_connection.close()
subprocess.run([PG_BIN / "pg_ctl", "-D", PG_DATA_DIR, "-m", "fast", "stop"], check=False,
               capture_output=True)
print(f"{RULE}\nError analysis: QS3/QS4 failures of the selected configuration\n{RULE}")
for _cause, _count in Counter(f["cause"] for f in failures).most_common():
    print(f"  {_cause:<20} {_count:>3}")
for _failure in failures[:5]:
    print(f"  - {_failure['query_text']}  →  target {_failure['target_names']}  "
          f"({_failure['cause']})")
print(f"{RULE}\n  {len(failures)} of {sum(q.row['set'] in ('QS3', 'QS4') for q in queries)} "
      f"cross-script queries failed")

────────────────────────────────────────────────────────────────────────
Error analysis: QS3/QS4 failures of the selected configuration
────────────────────────────────────────────────────────────────────────
  ranking_error         24
  transcription_error    6
  missing_equivalence    1
  - אב הורוויץ  →  target Valentin Hurvitz / אב הורוויץ  (transcription_error)
  - מרדכי אריה יאימאוויטש  →  target Hector Leonardo Jaimovich / מרדכי אריה יאימאוויטש  (ranking_error)
  - שפרינצה סאדיגורסקי  →  target Susana Sadigursky / שפרינצה סאדיגורסקי  (ranking_error)
  - אב הורוויץ  →  target Valentin Hurvitz / אב הורוויץ  (transcription_error)
  - הינדי אקסמן  →  target Rosa Ocsman / הינדי אקסמן  (transcription_error)
────────────────────────────────────────────────────────────────────────
  31 of 128 cross-script queries failed


**7. Outputs and Consistency Check**

This cell writes the results of Experiment 4 and then reads them back from disk to check them independently of the cells above:

1. `exp4_test_metrics.csv`, `exp4_mcnemar.json`, `exp4_latency.json`, `exp4_failures.csv` and one per-query file per configuration, and the overall and per-set rows of each configuration in `metrics.csv` with `split = test`
2. each configuration's overall R@10, recomputed from its per-query file as the macro-average over query sets, equals the reported one, and every McNemar count equals a recount of the per-query hits
3. every failure listed is a QS3 or QS4 query the selected configuration missed, and every such query is listed

It ends with the configurations measured and the result of the check.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Outputs and consistency check  —  Experiment 4's files, re-read from disk
# ─────────────────────────────────────────────────────────────────────────────
if "failures" not in globals():
    raise RuntimeError("Run every cell above first.")

write_csv(RESULTS_DIR / "exp4_test_metrics.csv", test_rows)
write_csv(RESULTS_DIR / "exp4_failures.csv", failures)
(RESULTS_DIR / "exp4_mcnemar.json").write_text(json.dumps(mcnemar_tests, indent=1) + "\n",
                                               encoding="utf-8")
(RESULTS_DIR / "exp4_latency.json").write_text(json.dumps(
    latency | {"e4_fine_tuning": e4_report, "frozen_commit": FROZEN_COMMIT}, indent=1) + "\n",
    encoding="utf-8")
for _role, _entry in ROLES.items():
    write_csv(RESULTS_DIR / "per_query" / f"exp4_{_entry['spec']['config_id']}.csv", [
        {"query_id": q.row["query_id"], "set": q.row["set"], "r_at_10": round(m_r10, 4),
         "target_in_top10": int(m_hit), "latency_ms": round(ms, 2),
         "top10": ";".join(f"{person_ids[p]}:{s:.3f}" for p, s in zip(ids[:10], scores[:10])
                           if p >= 0 and s >= _entry["params"]["tau_d"])}
        for q, m_r10, m_hit, ms, ids, scores in zip(
            queries, query_metrics[_role]["r10"], query_metrics[_role]["hit"],
            answers[_role]["ms"], answers[_role]["ids"], answers[_role]["scores"])])
_dev_rows = [r for r in read_csv(RESULTS_DIR / "metrics.csv") if r["split"] != "test"]
_columns = list(_dev_rows[0])
for _role, _metrics in query_metrics.items():
    _spec = ROLES[_role]["spec"]
    for _set_name in (*QUERY_SETS, "macro"):
        _rows = np.ones(len(queries), bool) if _set_name == "macro" else (
            query_set_index == QUERY_SETS.index(_set_name))
        _r10, _rr = macro_with_interval(_metrics["r10"], _rows), macro_with_interval(_metrics["rr"],
                                                                                  _rows)
        _dev_rows.append({**{c: _spec.get(c, "") for c in _columns}, "config_hash":
                          ROLES[_role]["config_hash"], "params": json.dumps(ROLES[_role]["params"]),
                          "split": "test", "fold": "", "query_set": _set_name, "subset": "all",
                          "n_queries": int(_rows.sum()), "r_at_10": _r10[0], "r_at_10_lo": _r10[1],
                          "r_at_10_hi": _r10[2], "mrr": _rr[0], "mrr_lo": _rr[1], "mrr_hi": _rr[2],
                          "unrelated_at_10": float(_metrics["unrelated"][_rows].mean()),
                          "empty_rate": float(_metrics["empty"][_rows].mean()),
                          "latency_p50_ms": float(np.percentile(answers[_role]["ms"][_rows], 50)),
                          "latency_p95_ms": float(np.percentile(answers[_role]["ms"][_rows], 95)),
                          "carried_forward": "false", "status": "ok", "notes": f"exp4 {_role}"})
write_csv(RESULTS_DIR / "metrics.csv", _dev_rows, _columns)


def output_problems() -> list[str]:
    """Re-read the outputs and check them against a plain recount."""
    problems, hits = [], {}
    overall = {r["role"]: float(r["r_at_10"]) for r in read_csv(
        RESULTS_DIR / "exp4_test_metrics.csv") if r["stratum"] == "overall"}
    for role, entry in ROLES.items():
        rows = read_csv(RESULTS_DIR / "per_query" / f"exp4_{entry['spec']['config_id']}.csv")
        by_set = defaultdict(list)
        for row in rows:
            by_set[row["set"]].append(float(row["r_at_10"]))
        if abs(np.mean([np.mean(v) for v in by_set.values()]) - overall[role]) > 1e-4:
            problems.append(f"{role}: overall R@10 differs when recounted")
        hits[role] = {row["query_id"]: int(row["target_in_top10"]) for row in rows}
    stored = json.loads((RESULTS_DIR / "exp4_mcnemar.json").read_text(encoding="utf-8"))
    for other, test in stored.items():
        only = sum(hits["selected"][q] and not hits[other][q] for q in hits["selected"])
        if only != test["selected_only"]:
            problems.append(f"McNemar vs {other}: counts differ when recounted")
    missed = {q for q, hit in hits["selected"].items()
              if not hit and q.split("-")[0] in ("QS3", "QS4")}
    listed = {row["query_id"] for row in read_csv(RESULTS_DIR / "exp4_failures.csv")}
    return problems + ([] if missed == listed else ["failure list differs from the missed queries"])


_problems = output_problems()
print(f"{RULE}\nConsistency: Experiment 4 outputs in {RESULTS_DIR}/\n{RULE}")
print(f"  {'OK ' if not _problems else '!! '}outputs              "
      + ("consistent" if not _problems else f"{len(_problems)} problem(s)"))
for _problem in _problems:
    print(f"        - {_problem}")
print(f"{RULE}\n  {len(ROLES)} configurations measured on {len(queries)} test queries  "
      f"({'consistent' if not _problems else 'inconsistent'})")
_selected_r10 = next(r["r_at_10"] for r in test_rows
                     if r["role"] == "selected" and r["stratum"] == "overall")
print(f"\n✓ Confirmation complete (selected {ROLES['selected']['spec']['config_id']}, "
      f"test R@10 {_selected_r10:.3f}).")
print(f"  test outputs consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: Experiment 4 outputs in results/
────────────────────────────────────────────────────────────────────────
  OK outputs              consistent
────────────────────────────────────────────────────────────────────────
  3 configurations measured on 415 test queries  (consistent)

✓ Confirmation complete (selected exp3_K2_sum, test R@10 0.869).
  test outputs consistent: True
